In [5]:
with open('../../data/tinyshakespeare.txt', 'r', encoding='utf-8') as f: 
    text = f.read()

chars = sorted(list(set(text)))

stoi = { s: i + 1 for (i,s) in enumerate(chars)}
itos = { i+1 : s for (i,s) in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])



In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import random



data = torch.tensor(encode(text), dtype=torch.long)
#create train and val splits
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
n_embd = 32 #embedding dimension

block_size = 8 #context length for predictions
batch_size = 32
epochs = 1000
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

def get_batch(split) : 
    data = train_data if split == 'train' else val_data
    idx = torch.randint(len(data) - block_size, (batch_size, ) )

    xb = torch.stack([data[i: block_size+ i]for i in idx]).to(device)
    yb = torch.stack([data[i+1 : block_size+ i + 1]for i in idx]).to(device)

    return xb, yb 


In [ ]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C
        x = tok_emb + pos_emb # (B,T,C)
        logits = self.lm_head(x) # (B,T,vocab_size)
        


        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :] # focus only on the last time step
            probs = F.softmax(logits, dim=-1) # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            idx = torch.cat((idx, idx_next), dim=1) # append sampled index to the running sequence
        return idx


model = BigramLanguageModel(vocab_size=len(stoi)+1) # +1 for padding index 0
m = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# train model : 

for _ in range(epochs):
    xb, yb  = get_batch('train')
    
    optimizer.zero_grad(set_to_none=True)
    logits, loss = model(xb,yb)
    loss.backward()
    optimizer.step()

    if _%100 == 0 : 
        print(loss.item())




4.6537017822265625
4.57560396194458
4.426269054412842
4.305894374847412
4.17486572265625
4.059028148651123
3.978560447692871
3.873844623565674
3.7448105812072754
3.6900360584259033
